In [51]:
import pandas as pd
import numpy as np
import warnings
import unicodedata
from pathlib import Path
import re
from datetime import datetime
import random

# Configurar la opción para evitar la advertencia
pd.set_option('future.no_silent_downcasting', True)
2
# Leer el archivo original con TODAS las columnas como texto
archivo = Path('/Users/jhonvilcaranatintaya/postulantesmaestria.xlsx')
nombre_salida = archivo.with_stem(archivo.stem + "_final")
df = pd.read_excel(archivo, dtype=str)  # ← esta línea es clave

# Reemplazar "-" por NaN y evitar FutureWarning
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    df = df.replace("-", np.nan).infer_objects(copy=False)
    

# Mapeo de nombres antiguos ➝ nuevos
mapeo_columnas = {
    "Filial": "CODIGO_SEDE_FILIAL",
    "TIPO_PROCESO": "TIPO_PROCESO",
    "Semestre": "PROCESO_ADMISION",
    "Número de convocatoria": "NUMERO_CONVOCATORIA",
    "Tipo de Documento": "TIPO_DOCUMENTO",
    "N° Documento": "NRO_DOCUMENTO",
    "Nombres": "NOMBRES",
    "Primer Apellido": "PRIMER_APELLIDO",
    "Segundo Apellido": "SEGUNDO_APELLIDO",
    "Apellido de Casada": "APELLIDO_CASADA",
    "SOLO_UN_APELLIDO": "SOLO_UN_APELLIDO",
    "Género": "SEXO",
    "Fecha de Nacimiento": "FECHA_NACIMIENTO",
    "Pais de Nacimiento": "PAIS_NACIMIENTO",
    "Nacionalidad": "NACIONALIDAD",
    "Ubigeo Nacimiento": "UBIGEO_NACIMIENTO",
    "Ubigeo Domicilio": "UBIGEO_DOMICILIO",
    "Condición discapacidad": "CONDICION_DISCAPACIDAD",
    "Tipo discapacidad": "TIPO_DISCAPACIDAD",
    "Celular": "CELULAR",
    "Correo personal": "CORREO_PERSONAL",
    "Facultad a la que ingresó": "CODIGO_FACULTAD_UNIDAD",
    "Carrera/Escuela a la que ingresó": "CODIGO_PROGRAMA_OPCIONES",
    "Fecha Postulación": "FECHA_POSTULACION",
    "Puntaje Obtenido": "PUNTAJE_OBTENIDO",
    "MODALIDAD_ADMISION": "MODALIDAD_ADMISION",
    "Modalidad Estudios": "MODALIDAD_ESTUDIO",
    "Ingresante": "ES_INGRESANTE",
    "CODIGO_FACULTAD_UNIDAD_INGRESO": "CODIGO_FACULTAD_UNIDAD_INGRESO",
    "CODIGO_PROGRAMA_INGRESO": "CODIGO_PROGRAMA_INGRESO",
    "Fecha Ingreso": "FECHA_INGRESO",
    "Correo institucional": "CORREO_INSTITUCIONAL",
    "CODIGO_ORCID": "CODIGO_ORCID"
}

# Lista exacta y ordenada de columnas requeridas (finales)
columnas_ordenadas = [
    "CODIGO_SEDE_FILIAL", "TIPO_PROCESO", "PROCESO_ADMISION", "NUMERO_CONVOCATORIA",
    "TIPO_DOCUMENTO", "NRO_DOCUMENTO", "NOMBRES", "PRIMER_APELLIDO", "SEGUNDO_APELLIDO",
    "APELLIDO_CASADA", "SOLO_UN_APELLIDO", "SEXO", "FECHA_NACIMIENTO", "PAIS_NACIMIENTO",
    "NACIONALIDAD", "UBIGEO_NACIMIENTO", "UBIGEO_DOMICILIO", "CONDICION_DISCAPACIDAD",
    "TIPO_DISCAPACIDAD", "CELULAR", "CORREO_PERSONAL", "CODIGO_FACULTAD_UNIDAD",
    "CODIGO_PROGRAMA_OPCIONES", "FECHA_POSTULACION", "PUNTAJE_OBTENIDO", "MODALIDAD_ADMISION",
    "MODALIDAD_ESTUDIO", "ES_INGRESANTE", "CODIGO_FACULTAD_UNIDAD_INGRESO",
    "CODIGO_PROGRAMA_INGRESO", "FECHA_INGRESO", "CORREO_INSTITUCIONAL", "CODIGO_ORCID"
]


# Renombrar columnas que están en el mapeo
df.rename(columns=mapeo_columnas, inplace=True)

# Añadir columnas faltantes como vacías
for col in columnas_ordenadas:
    if col not in df.columns:
        df[col] = ""

# Reordenar columnas
df = df[columnas_ordenadas]


# Diccionarios de reemplazo
tipo_documento_map = {
    "DNI": 1,
    "PASAPORTE": 2,
    "CE": 3,
    "CÉDULA DE IDENTIDAD": 4,
    "DIDEXT": 5,
    "PERMISO TEMPORAL DE PERMANENCIA": 6,
    "CARNÉ DE IDENTIDAD": 7,
    "CÉDULA DE CIUDADANÍA": 8,
    "CARNÉ TEMPORAL DE PERMANENCIA": 9
}

codigo_sede_filial_map = {
    "PUNO": "F01",
    "SAN MARTÍN": "F02",
    "UNIVERSIDAD PERUANA UNIÓN SEDE LIMA": "S"
}

sexo_map = {
    "MASCULINO": 1,
    "FEMENINO": 2
}

# Función de reemplazo con manejo de mayúsculas
def map_with_fallback(value, mapping):
    if pd.isna(value):
        return value
    val_upper = str(value).strip().upper()
    return mapping.get(val_upper, value)

# Validación de fechas con formato dd/mm/aaaa
def validar_fecha_ddmmaaaa(fecha):
    try:
        fecha_obj = pd.to_datetime(str(fecha), dayfirst=True, errors='coerce')
        if pd.isnull(fecha_obj):
            return ""
        return fecha_obj.strftime('%d/%m/%Y')
    except:
        return ""

# Aplicar reemplazos
df["TIPO_DOCUMENTO"] = df["TIPO_DOCUMENTO"].apply(lambda x: map_with_fallback(x, tipo_documento_map))
df["CODIGO_SEDE_FILIAL"] = df["CODIGO_SEDE_FILIAL"].apply(lambda x: map_with_fallback(x, codigo_sede_filial_map))
df["SEXO"] = df["SEXO"].apply(lambda x: map_with_fallback(x, sexo_map))

# Formato obligatorio
df["TIPO_PROCESO"] = 1
df["NUMERO_CONVOCATORIA"] = 1
df["NUMERO_CONVOCATORIA"] = 1

# Validar formato de fechas
df["FECHA_NACIMIENTO"] = df["FECHA_NACIMIENTO"].apply(validar_fecha_ddmmaaaa)
df["FECHA_POSTULACION"] = df["FECHA_POSTULACION"].apply(validar_fecha_ddmmaaaa)



# Calcular si tiene solo un apellido
# Asegurar columna vacía
df["SOLO_UN_APELLIDO"] = ""

# Normalizar SEGUNDO_APELLIDO a string sin espacios
segundo_ape = df["SEGUNDO_APELLIDO"].astype(str).str.strip()

# Crear máscaras
mask_solo_apellido = segundo_ape == ""              # Vacíos o solo espacios
mask_doble_apellido = segundo_ape != ""             # No vacíos

# Asignar valores
df.loc[mask_solo_apellido, "SOLO_UN_APELLIDO"] = 1
df.loc[mask_doble_apellido, "SOLO_UN_APELLIDO"] = 0



    
pais_nacimiento_map = {"AFGANISTÁN": 9327, "ALBANIA": 9443, "ALEMANIA": 9402, "ALEMANIA REPÚBLICA DEMOCRÁTICA": 9401, 
                       "ANDORRA, PRINCIPADO DE": 9442, "ANGOLA": 9113, "ANTIGUA Y BARBUDA": 9237, "ANTILLAS HOLANDESAS": 9201, 
                       "ARABIA SAUDITA": 9319, "ARGELIA": 9101, "ARGENTINA": 9202, "AUSTRALIA": 9501, "AUSTRIA": 9403, "AZERBAIYÁN": 9446, 
                       "BAHAMAS, ISLAS": 9203, "BANGLADESH": 9334, "BARBADOS": 9229, "BAREIN": 9341, "BÉLGICA": 9404, "BELICE": 9232, 
                       "BIELORRUSIA": 9436, "BIRMANIA": 9340, "BOLIVIA": 9204, "BOSNIA Y HERZEGOVINA": 9445, "BOTSUANA": 9130, 
                       "BRASIL": 9205, "BRUNÉI": 9345, "BULGARIA": 9405, "BURKINA FASO": 9114, "CAMBOYA": 9333, "CAMERÚN": 9123, 
                       "CANADÁ": 9206, "CARIBE NEERLANDÉS": 9241, "CATAR": 9338, "CHILE": 9210, "CHINA, REPÚBLICA POPULAR": 9302, 
                       "CHIPRE": 9407, "COLOMBIA": 9207, "COREA DEL NORTE": 9316, "COSTA DE MARFIL": 9102, "COSTA RICA": 9208, 
                       "CROACIA": 9431, "CUBA": 9209, "DINAMARCA": 9408, "DOMINICA": 9234, "ECUADOR": 9211, "EL SALVADOR": 9212, 
                       "EMIRATOS ÁRABES UNIDOS": 9337, "ESLOVAQUIA": 9438, "ESLOVENIA": 9433, "ESPAÑA": 9409, 
                       "ESTADOS UNIDOS DE AMÉRICA": 9213, "ESTONIA": 9444, "ETIOPÍA": 9120, "FILIPINAS": 9332, "FINLANDIA": 9410, 
                       "FRANCIA": 9411, "GABÓN": 9131, "GEORGIA": 9331, "GHANA": 9112, "GRANADA": 9240, "GRECIA": 9413, "GUATEMALA": 9215, 
                       "GUAYANA FRANCESA": 9230, "GUINEA ECUATORIAL": 9122, "GUYANA": 9231, "HAITI": 9216, "HONDURAS": 9217, 
                       "HONG KONG (COLONIA BRITÁNICA)": 9303, "HUNGRÍA": 9415, "INDIA": 9304, "INDONESIA": 9311, "IRAK": 9305, "IRÁN": 9324, 
                       "IRLANDA": 9418, "IRLANDA DEL NORTE": 9441, "ISLA DE JERSEY": 9453, "ISLANDIA": 9416, "ISLAS BERMUDAS": 9238, 
                       "ISLAS CAIMÁN": 9236, "ISRAEL": 9306, "ITALIA": 9417, "JAMAICA": 9218, "JAPÓN": 9307, "JORDANIA": 9313, 
                       "KAZAJSTÁN": 9321, "KENIA": 9104, "KIRGUISTÁN": 9326, "KUWAIT": 9328, "LETONIA": 9439, "LÍBANO": 9308, 
                       "LIBERIA": 9141, "LIBIA": 9116, "LIECHTENSTEIN, PRINCIPADO DE": 9419, "LITUANIA": 9440, 
                       "LUXEMBURGO, GRAN DUCADO DE": 9420, "MACEDONIA": 9437, "MADAGASCAR": 9119, "MALASIA": 9330, "MALTA": 9421, 
                       "MARRUECOS": 9105, "MAURICIO": 9129, "MÉXICO": 9219, "MICRONESIA": 9506, "MOLDAVIA": 9432, 
                       "MÓNACO, PRINCIPADO DE": 9422, "MONGOLIA": 9317, "MONTENEGRO": 9452, "MOZAMBIQUE": 9117, "NAMIBIA": 9132, 
                       "NEPAL": 9336, "NICARAGUA": 9220, "NIGERIA": 9111, "NORUEGA": 9423, "NUEVA ZELANDA": 9502, "OMÁN": 9339, 
                       "PAÍSES BAJOS": 9414, "PAKISTÁN": 9318, "PALAU": 9505, "PALESTINA": 9314, "PANAMA": 9221, "PAPÚA NUEVA GUINEA": 9503, 
                       "PARAGUAY": 9222, "PERÚ": 9233, "POLONIA": 9424, "PORTUGAL": 9425, "PUERTO RICO": 9223, 
                       "R. UNIDO DE G. BRETAÑA E IRLANDA DEL NTE.": 9450, "REINO UNIDO": 9412, 
                       "REP. DEMOCRÁTICA DE SANTO TOME Y PRÍNCIPE": 9143, "REP. DEMOCRÁTICA DEL CONGO": 9124, 
                       "REPÚBLICA ÁRABE DE EGIPTO": 9103, "REPÚBLICA CHECA": 9406, "REPÚBLICA DE ARMENIA": 9346, 
                       "REPÚBLICA DE BURUNDI": 9125, "REPÚBLICA DE CABO VERDE": 9136, "REPUBLICA DE COREA DEL SUR": 9301, 
                       "REPÚBLICA DE FIYI": 9508, "REPÚBLICA DE GAMBIA": 9134, "REPÚBLICA DE GUINEA": 9127, 
                       "REPÚBLICA DE GUINEA - BISAU": 9142, "REPÚBLICA DE KIRIBATI": 9507, "REPÚBLICA DE LAS SEYCHELLES": 9147, 
                       "REPÚBLICA DE NÍGER": 9138, "REPÚBLICA DE RUANDA": 9144, "REPÚBLICA DE UGANDA": 9140, "REPÚBLICA DEL CONGO": 9133, 
                       "REPÚBLICA DEMOCRÁTICA DE TIMOR ORIENTAL": 9347, "REPÚBLICA DEMOCRÁTICA POPULAR LAO": 9349, 
                       "REPÚBLICA DOMINICANA": 9224, "REPÚBLICA ISLÁMICA DE MAURITANIA": 9145, "REPÚBLICA TOGOLESA": 9146, 
                       "REPÚBLICA UNIDA DE TANZANIA": 9126, "RUMANIA": 9426, "RUSIA": 9429, "SAMOA AMERICANA": 9504, 
                       "SAN CRISTOBAL Y NIEVES": 9239, "SAN MARINO, REPÚBLICA DE": 9447, "SAN VICENTE Y LAS GRANADINAS": 9242, 
                       "SANTA LUCÍA": 9235, "SENEGAL": 9121, "SERBIA": 9448, "SIERRA LEONA": 9135, "SINGAPUR": 9325, "SIRIA": 9312, 
                       "SOMALIA": 9110, "SRI LANKA": 9309, "SUDÁFRICA": 9106, "SUDÁN": 9128, "SUECIA": 9427, "SUIZA": 9428, 
                       "SURINAM": 9225, "SWAZILANDIA": 9118, "TAILANDIA": 9310, "TAIWÁN": 9323, "TAYIKISTÁN": 9342, 
                       "TRINIDAD Y TOBAGO": 9226, "TÚNEZ": 9109, "TURKMENISTÁN": 9329, "TURQUÍA": 9315, "UCRANIA": 9434, 
                       "UNIÓN DE LAS COMORAS": 9139, "URUGUAY": 9227, "UZBEKISTÁN": 9343, "VENEZUELA": 9228, "VIETNAM": 9320, 
                       "YEMEN": 9322, "YUGOSLAVIA": 9430, "ZAIRE": 9115, "ZAMBIA": 9107, "ZIMBABWE": 9108}


# Diccionario de nacionalidades (DESCRIPCIÓN -> CÓDIGO)
nacionalidad_map = {"AFGANO": 1, "ALBANÉS": 2, "ALEMANO": 3, "ANDORRANO": 4, "ANGOLEÑO": 5, "ANTIGUANO": 6, "ARGELINO": 8, "ARGENTINO": 9,
                    "ARMENIO": 10, "AUSTRALIANO": 12, "AUSTRIACO": 13, "AZERBAIYANO": 14, "BAHAMENSE": 15, "BANGLADESHÍ": 16, 
                    "BARBADENSE": 17, "BAREINÍ": 18, "BELGA": 19, "BELICEÑO": 20, "BIELORRUSO": 22, "BIRMANO": 23, "BOLIVIANO": 24,
                    "BRASILEÑO": 27, "BRITÁNICO": 149, "BRUNEANO": 28, "BÚLGARO": 29, "BURKINÉS": 30, "BURUNDÉS": 31, "BUTANÉS": 32,
                    "CABOVERDIANO": 33, "CAMBOYANO": 34, "CAMERUNÉS": 35, "CANADIENSE": 36, "CATARÍ": 37, "CEILANÉS": 173,
                    "CENTROAFRICANO": 150, "CHADIANO": 38, "CHECO": 151, "CHILENO": 39, "CHINO": 40, "CHIPRIOTA": 41, "COLOMBIANO": 43,
                    "COMORENSE": 44, "CONGOLEÑO": 154, "COOKIANO": 90, "COSTARRICENSE": 48, "CRISTOBALEÑO": 161, "CROATA": 49, "CUBANO": 50,
                    "DANÉS": 51, "DOMINICANO": 155, "DOMINIQUÉS": 52, "ECUATOGUINEANO": 77, "ECUATORIANO": 53, "EGIPCIO": 54, "EMIRATÍ": 56,
                    "ERITREO": 57, "ESLOVACO": 58, "ESLOVENO": 59, "ESPAÑOL": 60, "ESTADOUNIDENSE": 61, "ESTONIO": 62, "ETÍOPE": 63,
                    "FILIPINO": 64, "FINLANDÉS": 65, "FIYIANO": 66, "FRANCÉS": 67, "GABONÉS": 68, "GAMBIANO": 69, "GEORGIANO": 70,
                    "GHANÉS": 72, "GIBRALTAREÑO": 71, "GRANADINO": 73, "GRIEGO": 74, "GROENLANDÉS": 75, "GUATEMALTECO": 76, "GUINEANO": 79,
                    "GUYANÉS": 80, "HAITIANO": 81, "HINDÚ": 84, "HONDUREÑO": 82, "HÚNGARO": 83, "INDONESIO": 145, "IRANÍ": 87, "IRAQUÍ": 86,
                    "IRLANDÉS": 88, "ISLANDÉS": 89, "ISRAELÍ": 93, "ITALIANO": 94, "JAMAICANO": 95, "JAPONÉS": 96, "JORDANO": 97, 
                    "KAZAJO": 98, "KENIATA": 99, "KIRGUISTÁN": 100, "KIRIBATIANO": 101, "KUWAITÍ": 102, "LAOSIANO": 103, "LESOTENSE": 104,
                    "LETÓN": 105, "LIBANÉS": 106, "LIBERIANO": 107, "LIBIO": 108, "LIECHTENSTEINIANO": 109, "LITUANO": 110, 
                    "LUXEMBURGUEÑO": 111, "MACEDONIO": 152, "MALASIO": 113, "MALAUÍ": 114, "MALDIVIO": 115, "MALGACHE": 112,
                    "MALIENSE": 116, "MALTÉS": 117, "MARFILEÑO": 47, "MARROQUÍ": 118, "MARSHALÉS": 91, "MARTINIQUÉS": 119,
                    "MAURICIANO": 120, "MAURITANO": 121, "MEXICANO": 122, "MICRONESIO": 123, "MOLDAVO": 124, "MONEGASCO": 125,
                    "MONGOL": 126, "MONTENEGRINO": 127, "MOZAMBIQUEÑO": 128, "NAMIBIO": 129, "NAURUANO": 130, "NEERLANDÉS": 138,
                    "NEOZELANDÉS": 136, "NEPALÍ": 131, "NICARAGÜENSE": 132, "NIGERIANO": 134, "NIGERINO": 133, "NORCOREANO": 45,
                    "NORUEGO": 135, "OMANÍ": 137, "PAKISTANÍ": 139, "PALAUANO": 140, "PALESTINO": 141, "PANAMEÑO": 142, "PAPÚ": 143,
                    "PARAGUAYO": 144, "PERUANO": 85, "POLACO": 146, "PORTUGUÉS": 147, "PUERTORRIQUEÑO": 148, "RUANDÉN": 157, "RUMANO": 158,
                    "RUSO": 159, "SALOMONENSE": 92, "SALVADOREÑO": 55, "SAMOANO": 160, "SANMARINENSE": 162, "SANTALUCENSE": 164, 
                    "SANTOTOMENSE": 165, "SANVICENTINO": 163, "SAUDÍ": 7, "SENEGALÉS": 166, "SERBIO": 167, "SEYCHELENSE": 168, 
                    "SIERRALEONÉS": 169, "SINGAPUREÑO": 170, "SIRIO": 171, "SOMALÍ": 172, "SUAZI": 174, "SUDAFRICANO": 156, "SUDANÉS": 176,
                    "SUECO": 177, "SUIZO": 178, "SURCOREANO": 46, "SURINAMÉS": 179, "SURSUDANÉSO": 175, "TAILANDÉSO": 180, "TAIWANÉS": 202,
                    "TANZANO": 181, "TAYIKO": 182, "TIMORENSE": 183, "TOGOLÉSO": 184, "TONGANO": 185, "TRINITENSE": 186, "TUNECINO": 187,
                    "TURCO": 189, "TURCOMANO": 188, "TUVALUANO": 190, "UCRANIANO": 191, "UGANDÉS": 192, "URUGUAYO": 193, "UZBEKO": 194, 
                    "VANUATUENSE": 195, "VATICANO": 42, "VENEZOLANO": 196, "VIETNAMITA": 197, "YEMENÍ": 198, "YIBUTIANO": 199, 
                    "ZAMBIANO": 200, "ZIMBABUENSE": 201}


# Función para reemplazar con mayúsculas y fallback
def map_with_fallback(value, mapping):
    if pd.isna(value):
        return value
    val_upper = str(value).strip().upper()  # Convertir a mayúsculas
    val_without_accents = val_upper  # Si es necesario, elimina acentos
    return mapping.get(val_without_accents, value)

# Aplicar reemplazos a la columna "PAIS_NACIMIENTO"
df["PAIS_NACIMIENTO"] = df["PAIS_NACIMIENTO"].apply(lambda x: map_with_fallback(x, pais_nacimiento_map))

# Aplicar reemplazos a la columna "NACIONALIDAD"
df["NACIONALIDAD"] = df["NACIONALIDAD"].apply(lambda x: map_with_fallback(x, nacionalidad_map))

#Remplazar datos (CONDICION_DISCAPACIDAD, TIPO_DISCAPACIDAD) ACTUALIZAR CONTENIDO DE DISCAPACIDADES

# Función para normalizar texto (quita tildes y convierte a mayúsculas)
def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

# Diccionario de tipos de discapacidad (DESCRIPCIÓN -> CÓDIGO)
tipo_discapacidad_map = {
    "DISCAPACIDAD MOTRIZ": 1,
    "DISCAPACIDAD VISUAL": 2,
    "VISUAL Y ESQUEMA CORPORAL": 3,
    "DISMINUIDOS VISUALES": 4,
    "DISCAPACIDAD AUDITIVA": 5,
    "AUTISMO": 6,
    "DISCAPACIDAD MENTAL": 7,
    "PARALISIS CEREBRAL": 8,
    "DISCAPACIDAD INTELECTUAL": 9,
    "SORDOCEGUERA": 10,
    "NO CUENTA CON INFORMACION": 11,
    "OTROS": 12,
    "SINDROME DE ASPERGER": 13,
    "HEMIPLEGIA NO IDENTIFICADA": 14,
    "ESTENOSIS CONGENITA DE LA VALVULA AORTICA": 15,
    "MULTIDISCAPACIDAD": 16,
    "DISCAPACIDAD FISICA": 17,
    "TRANSTORNO DEL ESPECTRO AUTISTA": 18,
    "T. POR DEFICIT DE ATENCION CON HIPERACTIVIDAD": 19,
    "T. ESPECIFICO DEL APRENDIZAJE": 20,
    "T. MENTALES Y DEL COMPORTAMIENTO": 21,
    "ENFERMEDADES RARAS": 22,
    "TALLA BAJA": 23,
    "TALENTO": 24,
    "SUPERDOTACION": 25
}

# 🔹 Normalizar y mapear CONDICION_DISCAPACIDAD ("SI"->1, "NO"->0)
df["CONDICION_DISCAPACIDAD"] = df["CONDICION_DISCAPACIDAD"].apply(normalizar_texto).map({"SI": 1, "NO": 0})

# 🔹 Normalizar y mapear TIPO_DISCAPACIDAD
df["TIPO_DISCAPACIDAD"] = df["TIPO_DISCAPACIDAD"].apply(normalizar_texto).apply(lambda x: tipo_discapacidad_map.get(x, x))

# 🔹 Si la condición de discapacidad es 0 (NO), se borra el tipo de discapacidad
df.loc[df["CONDICION_DISCAPACIDAD"] == 0, "TIPO_DISCAPACIDAD"] = ""



#Remplazo CODIGO_PROGRAMA_OPCIONES (LOS NOMBRES NO DEBEN TENER TILDE EN EL CODIGO YA QUE ESTAN NORMALIZADOS PARA SU REMPLAZO)

# Limpiar los datos (eliminar espacios y normalizar caracteres)
df['CODIGO_PROGRAMA_OPCIONES'] = df['CODIGO_PROGRAMA_OPCIONES'].apply(lambda x: unicodedata.normalize('NFKD', str(x)).encode('ASCII', 'ignore').decode('ASCII'))

# Asegurarse de que la columna 'CODIGO_PROGRAMA_INGRESO' es de tipo str
df['CODIGO_PROGRAMA_OPCIONES'] = df['CODIGO_PROGRAMA_OPCIONES'].astype(str)

# Diccionario de reemplazo
reemplazos = {
    #SEGUNDAS ESPECIALIDADES
    "Posgrado de Segunda Especialidad de Enfermeria en Emergencias y Desastres": 67,
    "Posgrado de Segunda Especialidad de Enfermeria en Gineco Obstetricia": 81, #Cambiar a 81 si es 1er ciclo o mantener 244
    "Posgrado de Segunda Especialidad en Enfermeria en Administracion y Gestion": 63,
    "Posgrado de Segunda Especialidad en Enfermeria en Centro Quirurgico": 73,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Neonatales": 75,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Pediatricos": 76,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos": 77,
    "Posgrado de Segunda Especialidad en Enfermeria en Neurologia y Neurocirugia": 82,
    "Posgrado de Segunda Especialidad en Estadistica Aplicada para Investigacion": 87,
    "Posgrado de Segunda Especialidad de Enfermeria en Oncologia": 83,
    "Posgrado de Segunda Especialidad de Enfermeria en Oncologia": 83,
    "Posgrado de Segunda Especialidad en Enfermeria en Pediatria": 84,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Traumatologia y Ortopedia": 79,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanastesica": 220,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos": 66,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Posgrado de Segunda Especialidad en Salud Ocupacional": 91,
    "Posgrado de Segunda Especialidad en Enfermeria en Cardiologia": 64,
    "Posgrado de Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Posgrado de Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80, 
    "Segunda Especialidad de Enfermeria en Cardiologia": 64,
    "Segunda Especialidad de Enfermeria en Cirugia Cardiovascular": 65,
    "Segunda Especialidad de Enfermeria en Cuidados Intensivos": 66,
    "Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Segunda Especialidad de Enfermeria en Nefrologia": 69,
    "Segunda Especialidad de Enfermeria en Pediatria": 84,
    "Segunda Especialidad de Enfermeria en Perfusion Extracorporea y Asistencia Circulatoria": 85,
    "Segunda Especialidad de Enfermeria en Cuidado de la Persona con Diabetes con Mencion en Educacion en Diabetes": 269,
    "Segunda Especialidad en Antiguo Testamento": 71,
    "Segunda Especialidad en Asesoria Genetica en Enfermeria Oncologica": 72,
    "Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con Mencion en Traumatologia y Ortopedia": 79,
    "Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanestesica": 220,
    "Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Segunda Especialidad en Enfermeria en Gineco Obstetricia": 81,
    "Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80,
    "Segunda Especialidad en Enfermeria en Psicologia Clinica y de la Salud": 222,
    "Segunda Especialidad en Psicologia Clinica y de la Salud": 222,
    "Segunda Especialidad en Estudios Adventistas": 88,
    "Segunda Especialidad en Nuevo Testamento": 89,
    "Segunda Especialidad en Profecias Biblicas": 90,
    "Segunda Especialidad en Salud Ocupacional": 91,
    "Segunda Especialidad Profesional en Neuropsicologia": 272,
    "Segunda Especialidad Profesional en Psicooncologia y Cuidados Paliativos": 258,

    # MAESTRÍAS
    "Maestria Bim Facility Management": 273,
    "Maestria en Administracion de Negocios con mencion en Gestion Empresarial": 246,
    "Maestria en Auditoria con mencion en Auditoria Integral": 40,
    "Maestria en Ciencias de la Familia": 238,
    "Maestria en Educacion con mencion en Curriculo": 230,
    "Maestria en Ingenieria Ambiental": 48,
    "Maestria en Ingenieria Ambiental y Desarrollo Sostenible": 247,
    "Maestria en Ingenieria de Sistemas con mencion en Ingenieria de Software": 231,
    "Maestria en Liderazgo Eclesiastico": 50,
    "Maestria en Misionologia": 51,
    "Maestria en Nutricion Humana con Mencion en Alimentacion Basada en Plantas": 270,
    "Maestria en Nutricion Humana con mencion en Nutricion Clinica": 52,
    "Maestria en Nutricion Humana con mencion en Nutricion Vegetariana": 232,
    "Maestria en Teologia": 56,
    "Posgrado de Maestria en Teologia": 56,
    "Maestria en Teologia Biblica": 233,
    "Maestria en Teologia Historica": 234,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Finanzas": 226,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Liderazgo y Gestion Organizacional": 227,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Recursos Humanos": 39,
    "Posgrado de Maestria en Educacion: Administracion Educativa": 42,
    "Posgrado de Maestria en Educacion: Investigacion y Docencia Universitaria": 228,
    "Posgrado de Maestria en Educacion: Mencion en Psicologia Educativa": 229,
    "Posgrado de Maestria en Enfermeria con Mencion en Administracion y Gestion": 46,
    "Posgrado de Maestria en Ingenieria de Sistemas con mencion en Direccion y Gestion de Tecnologias de Informacion": 49,
    "Posgrado de Maestria en Psicologia Clinica y de la Salud": 53,
    "Posgrado de Maestria en Salud Publica Mencion: Gestion de los Servicios de Salud": 223,
    "Posgrado de Maestria en Salud Publica Mencion: Salud Colectiva y Promocion de la Salud": 224,
    "Posgrado de Maestria en Terapia Familiar y de Pareja": 41,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Administracion de Recursos Humanos": 39,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial ": 246,
    "Posgrado de Maestria en Ciencias de la Familia Mencion: Terapia Familiar": 41,
    "Posgrado de Maestria en Ingenieria Ambiental": 48,
    "Posgrado de Maestria en Ingenieria de Sistemas : Direccion y Gestion de TI": 49,
    "Posgrado de Maestria en Misionologia": 51,
    "Posgrado de Maestria en Auditoria Mencion: Auditoria Integral": 40,
    "Posgrado de Maestria en Nutricion Humana con Mencion en Nutricion Vegetariana": 232,
        "Posgrado de Maestria en Teologia Biblica": 233,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial": 246,

    # DOCTORADOS
    "Doctorado en Administracion de Negocios": 11,
    "Doctorado en Ciencias Contables y Financieras": 276,
    "Posgrado de Doctorado en Educacion mencion: Gestion Educativa": 235,
    "Doctorado en Educacion con mencion en Curriculo y Docencia": 12,
    "Posgrado de Doctorado en Enfermeria": 13,
    "Doctorado en Ingenieria de Sistemas": 14,
    "Posgrado de Doctorado en Salud Publica": 15,
    "Doctorado en Teologia": 16,
    "Posgrado de Doctorado en Teologia": 16,
    "Posgrado de Doctorado en Administracion de Negocios": 11,
    "Posgrado de Doctorado en Educacion mencion: Curriculo y Docencia": 12,
}

# Realizar el reemplazo en la columna 'CODIGO_PROGRAMA_INGRESO'
df['CODIGO_PROGRAMA_OPCIONES'] = df['CODIGO_PROGRAMA_OPCIONES'].replace(reemplazos)


#Remplazar CODIGO_FACULTAD_UNIDAD, MODALIDAD_ESTUDIO , 
# Diccionario de facultades/unidades
facultad_unidad_map = {
    "Ciencias de la Salud": 7,
    "Ciencias Empresariales": 6,
    "Ingeniería y Arquitectura": 3,
    "Escuela General de Posgrado": 9,
    "No Consignado": 2,
    "Ciencias Humanas y Educación": 5,
    "Teología": 1
}

# Diccionario de modalidad de estudio
reemplazos_modalidad = {
    "Dato no declarado": 0,
    "Presencial": 1,
    "Semipresencial": 2,
    "A Distancia": 3
}

# Función para reemplazo seguro con fallback
def map_with_fallback(value, mapping):
    if pd.isna(value):
        return value
    return mapping.get(value, value)

# Función para normalizar texto (quita tildes y pasa a mayúsculas)
def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

# Aplicar reemplazos
df["CODIGO_FACULTAD_UNIDAD"] = df["CODIGO_FACULTAD_UNIDAD"].apply(lambda x: map_with_fallback(x, facultad_unidad_map))
df["MODALIDAD_ESTUDIO"] = df["MODALIDAD_ESTUDIO"].replace(reemplazos_modalidad)

# Asignar todos los valores a "8" en MODALIDAD_ADMISION
df["MODALIDAD_ADMISION"] = 8

# Normalizar y reemplazar valores en ES_INGRESANTE
df["ES_INGRESANTE"] = df["ES_INGRESANTE"].apply(normalizar)
df["ES_INGRESANTE"] = df["ES_INGRESANTE"].replace({
    "INGRESANTE": 1,
    "NO_INGRESANTE": 0
})

#RELLENAR LAS COLUMNAS CODIGO_PROGRAMA_OPCIONES Y CODIGO_PROGRAMA_INGRESO
# Asegurar que las columnas existen
columnas = [
    "ES_INGRESANTE",
    "CODIGO_FACULTAD_UNIDAD", "CODIGO_PROGRAMA_OPCIONES",
    "CODIGO_FACULTAD_UNIDAD_INGRESO", "CODIGO_PROGRAMA_INGRESO"
]

if all(col in df.columns for col in columnas):
    # Convertir a string sin generar 'nan' como texto
    df["CODIGO_FACULTAD_UNIDAD"] = df["CODIGO_FACULTAD_UNIDAD"].fillna("").astype(str)
    df["CODIGO_PROGRAMA_OPCIONES"] = df["CODIGO_PROGRAMA_OPCIONES"].fillna("").astype(str)

    # Corregido: convertir ES_INGRESANTE a string
    mask = df["ES_INGRESANTE"].astype(str).str.strip() == "1"

    df.loc[mask & (df["CODIGO_FACULTAD_UNIDAD"].str.strip() != ""),
           "CODIGO_FACULTAD_UNIDAD_INGRESO"] = df["CODIGO_FACULTAD_UNIDAD"]

    df.loc[mask & (df["CODIGO_PROGRAMA_OPCIONES"].str.strip() != ""),
           "CODIGO_PROGRAMA_INGRESO"] = df["CODIGO_PROGRAMA_OPCIONES"]

#Remplazo de PUNTAJE_OBTENIDO = 0 y normalizacion de CELULAR, NRO_DOCUMENTO, UBIGEO_NACIMIENTO y UBIGEO_DOMICILIO
# === Normalizar CELULAR ===
def normalizar_celular(numero):
    if pd.isnull(numero):
        return ""
    numero = re.sub(r'\D', '', str(numero))
    if numero.startswith("51"):
        numero = numero[2:]
    if 9 <= len(numero) <= 12:
        return numero
    return ""

df['CELULAR'] = df['CELULAR'].apply(normalizar_celular)

# === Normalizar UBIGEOs a 6 dígitos solo si hay valor ===
def normalizar_ubigeo(valor):
    if pd.isna(valor) or str(valor).strip() == "" or str(valor).lower() == "nan":
        return ""
    return str(valor).zfill(6)

if 'UBIGEO_NACIMIENTO' in df.columns:
    df['UBIGEO_NACIMIENTO'] = df['UBIGEO_NACIMIENTO'].apply(normalizar_ubigeo)

if 'UBIGEO_DOMICILIO' in df.columns:
    df['UBIGEO_DOMICILIO'] = df['UBIGEO_DOMICILIO'].apply(normalizar_ubigeo)

# === Rellenar PUNTAJE_OBTENIDO con "0" si ES_INGRESANTE == "0" y está vacío ===
if "ES_INGRESANTE" in df.columns and "PUNTAJE_OBTENIDO" in df.columns:
    df["PUNTAJE_OBTENIDO"] = df["PUNTAJE_OBTENIDO"].astype(str)
    df.loc[
        (df["ES_INGRESANTE"] == "0") &
        (df["PUNTAJE_OBTENIDO"].str.strip() == ""),
        "PUNTAJE_OBTENIDO"
    ] = "0"

# === Reemplazo de facultad/programa solo si ES_INGRESANTE == "1" ===
if all(col in df.columns for col in [
    "ES_INGRESANTE", "CODIGO_FACULTAD_UNIDAD", "CODIGO_FACULTAD_UNIDAD_INGRESO",
    "CODIGO_PROGRAMA_OPCIONES", "CODIGO_PROGRAMA_INGRESO"
]):
    ingresantes = df["ES_INGRESANTE"] == "1"
    df.loc[ingresantes, "CODIGO_FACULTAD_UNIDAD"] = df.loc[ingresantes, "CODIGO_FACULTAD_UNIDAD_INGRESO"]
    df.loc[ingresantes, "CODIGO_PROGRAMA_OPCIONES"] = df.loc[ingresantes, "CODIGO_PROGRAMA_INGRESO"]


# VALIDAR SOLO SI EXISTE ALGÚN DOCUMENTO INVÁLIDO
reglas_documento_motivo = {
    1: lambda x: ("DNI debe tener exactamente 8 dígitos" if not (str(x).isdigit() and len(x) == 8) else None),
    2: lambda x: ("Pasaporte debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    3: lambda x: ("CE debe tener al menos 8 dígitos numéricos" if not (str(x).isdigit() and len(x) >= 8) else None),
    4: lambda x: ("Cédula de Identidad debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    5: lambda x: ("DIDEXT debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    6: lambda x: ("PTP debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    7: lambda x: ("Carné de Identidad debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    8: lambda x: ("Cédula de Ciudadanía debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    9: lambda x: ("Carné Temporal de Permanencia debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None)
}


# Asignar valores
df["SOLO_UN_APELLIDO"] = 0
df.loc[mask_solo_apellido, "SOLO_UN_APELLIDO"] = 1

# Mostrar advertencia y lista si hay casos
if mask_solo_apellido.sum() > 0:
    print(f"\n⚠️ Se detectaron {mask_solo_apellido.sum()} estudiantes con solo un apellido. Verificar información.")


#Validar celular y puntaje (Obligatorio)

# Limpieza y detección robusta de vacíos
df["CELULAR"] = df["CELULAR"].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})
df["PUNTAJE_OBTENIDO"] = df["PUNTAJE_OBTENIDO"].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})

# Máscaras actualizadas
mask_celular_vacio = df["CELULAR"] == ""
mask_puntaje_vacio = df["PUNTAJE_OBTENIDO"] == ""


# Normaliza y limpia correos
for col in ["CORREO_PERSONAL", "CORREO_INSTITUCIONAL", "ES_INGRESANTE"]:
    df[col] = df[col].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})

# Convertir ingresante a valor binario consistente
df["ES_INGRESANTE"] = df["ES_INGRESANTE"].replace({"1": "1"}).fillna("0")

# Máscaras de vacíos
mask_personal_vacio = df["CORREO_PERSONAL"] == ""
mask_institucional_vacio = (df["CORREO_INSTITUCIONAL"] == "") & (df["ES_INGRESANTE"] == "1")

# ==== 1. Si NO es ingresante, limpiar campos de ingreso ====
mask_no_ingresante = df["ES_INGRESANTE"].astype(str).str.strip() != "1"

campos_ingreso = [
    "CODIGO_FACULTAD_UNIDAD_INGRESO",
    "CODIGO_PROGRAMA_INGRESO",
    "FECHA_INGRESO",
    "CORREO_INSTITUCIONAL"
]

for col in campos_ingreso:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].astype(str) 
    df.loc[mask_no_ingresante, col] = ""

# ==== 2. Si PUNTAJE_OBTENIDO está vacío o nulo, asignar 0 ====
if "PUNTAJE_OBTENIDO" not in df.columns:
    df["PUNTAJE_OBTENIDO"] = ""

mask_puntaje_vacio = (
    df["PUNTAJE_OBTENIDO"].isna() |
    (df["PUNTAJE_OBTENIDO"].astype(str).str.strip() == "")
)

df.loc[mask_puntaje_vacio, "PUNTAJE_OBTENIDO"] = 0

# ==== 3. Si NACIONALIDAD = 85 y UBIGEO_NACIMIENTO vacío → asignar 140101 ====
if "NACIONALIDAD" not in df.columns:
    df["NACIONALIDAD"] = ""

if "UBIGEO_NACIMIENTO" not in df.columns:
    df["UBIGEO_NACIMIENTO"] = ""

mask_nac_peru = df["NACIONALIDAD"].astype(str).str.strip() == "85"
mask_ubigeo_vacio = (
    df["UBIGEO_NACIMIENTO"].isna() |
    (df["UBIGEO_NACIMIENTO"].astype(str).str.strip() == "")
)

df.loc[mask_nac_peru & mask_ubigeo_vacio, "UBIGEO_NACIMIENTO"] = "140101"

# ==== 4. Generar número de CELULAR aleatorio si está vacío ====
def generar_celular():
    return f"9{random.randint(10000000, 99999999)}"  # siempre inicia con 9 y tiene 9 dígitos

if "CELULAR" not in df.columns:
    df["CELULAR"] = ""

mask_celular_vacio = (
    df["CELULAR"].isna() |
    (df["CELULAR"].astype(str).str.strip() == "")
)

df.loc[mask_celular_vacio, "CELULAR"] = df.loc[mask_celular_vacio, "CELULAR"].apply(lambda _: generar_celular())

#GENERAR CORREOS INSTITUCIONALES A INGRESANTES

for col in ["CORREO_INSTITUCIONAL", "NOMBRES", "PRIMER_APELLIDO", "SEGUNDO_APELLIDO", "ES_INGRESANTE"]:
    if col not in df.columns:
        df[col] = ""

def _quitar_acentos_y_normalizar(s: str) -> str:
    """Quita tildes/diacríticos y deja solo letras en minúscula."""
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")  # sin acentos
    s = s.replace("ñ", "n")  # normaliza ñ
    s = re.sub(r"[^a-z]", "", s)  # solo letras
    return s

def _primer_token(s: str) -> str:
    """Devuelve el primer token 'palabra' (ej. primer nombre o primer apellido)."""
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""
    return s.split()[0]  # primer nombre / primer apellido

def _construir_correo_row(row) -> str:
    """Construye nombre.apellido@upeu.edu.pe con fallback al segundo apellido si falta el primero."""
    nom = _quitar_acentos_y_normalizar(_primer_token(row.get("NOMBRES", "")))
    ape = _quitar_acentos_y_normalizar(_primer_token(row.get("PRIMER_APELLIDO", "")))
    if not ape:  # fallback al segundo apellido si el primero está vacío
        ape = _quitar_acentos_y_normalizar(_primer_token(row.get("SEGUNDO_APELLIDO", "")))
    if not nom or not ape:
        return ""  # si no hay insumos, no forzamos correo
    return f"{nom}.{ape}@upeu.edu.pe"

# Normalizar la columna de correos a minúsculas, sin 'nan' textual
df["CORREO_INSTITUCIONAL"] = (
    df["CORREO_INSTITUCIONAL"]
    .astype(str).str.strip().str.lower()
    .replace({"nan": ""})
)

# Máscara de ingresantes (la columna puede estar como int o str en distintos pasos)
mask_ing = df["ES_INGRESANTE"].astype(str).str.strip() == "1"

# Define qué es un correo institucional válido
pat_institucional = r"^[a-z0-9._%+-]+@upeu\.edu\.pe$"
mask_valido = df["CORREO_INSTITUCIONAL"].str.match(pat_institucional, na=False)

# Filas a corregir: ingresantes con correo institucional vacío o de otro dominio
mask_a_corregir = mask_ing & ~mask_valido

# Construir y asignar
df.loc[mask_a_corregir, "CORREO_INSTITUCIONAL"] = (
    df.loc[mask_a_corregir].apply(_construir_correo_row, axis=1)
)

#GENERAR CORREOS PERSONAL A TODOS LOS POSTULANTES CON CORREOS INVALIDOS

# Asegurar columnas mínimas
for col in ["CORREO_PERSONAL", "NOMBRES", "PRIMER_APELLIDO", "SEGUNDO_APELLIDO", "NRO_DOCUMENTO"]:
    if col not in df.columns:
        df[col] = ""

# Limpiar columna
df["CORREO_PERSONAL"] = (
    df["CORREO_PERSONAL"] 
    .astype(str).str.strip().str.lower()
    .replace({"nan": "", "none": ""})
)

# Regex básico para validar formato de correo
pat_email_valido = r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"

# Lista de usuarios falsos a reemplazar aunque el correo sea válido
usuarios_invalidos = {"xxx", "xxxx", "otro", "correo", "default", "user", "correo1", "correo2", "xx", "sin"}

# Funciones auxiliares
def _quitar_acentos_y_normalizar(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFD", s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.replace("ñ", "n")
    s = re.sub(r"[^a-z]", "", s)
    return s

def _primer_token(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    return s.split()[0] if s else ""

def _usuario_placeholder(usuario: str) -> bool:
    return usuario in usuarios_invalidos

def _construir_correo_personal(row) -> str:
    nom = _quitar_acentos_y_normalizar(_primer_token(row.get("NOMBRES", "")))
    ape = _quitar_acentos_y_normalizar(_primer_token(row.get("PRIMER_APELLIDO", "")))
    if not ape:
        ape = _quitar_acentos_y_normalizar(_primer_token(row.get("SEGUNDO_APELLIDO", "")))

    doc = re.sub(r"\D", "", str(row.get("NRO_DOCUMENTO", "")))

    if nom and ape:
        user = f"{nom}.{ape}"
    elif nom:
        user = f"{nom}{doc[-2:] if doc else ''}"
    elif ape:
        user = f"{ape}{doc[-2:] if doc else ''}"
    elif doc:
        user = f"u{doc}"
    else:
        user = f"user{row.name}"

    return f"{user}@gmail.com"

# Separar usuario del correo actual
usuario_correo = df["CORREO_PERSONAL"].str.split("@").str[0]
dominio_correo = df["CORREO_PERSONAL"].str.split("@").str[1]

# Máscara de correos mal formados
mask_invalido = ~df["CORREO_PERSONAL"].str.match(pat_email_valido, na=False)

# Máscara de usuarios falsos aunque el correo tenga buen formato
mask_placeholder = usuario_correo.isin(usuarios_invalidos)

# Unimos todas las condiciones que implican que el correo debe ser corregido
mask_corregir = mask_invalido | mask_placeholder

# Aplicar construcción
df.loc[mask_corregir, "CORREO_PERSONAL"] = (
    df.loc[mask_corregir].apply(_construir_correo_personal, axis=1)
)

# ==== 5. Imprimir resumen ====
total_correcciones = (
    mask_puntaje_vacio.sum()
    + (mask_nac_peru & mask_ubigeo_vacio).sum()
    + mask_celular_vacio.sum()
    + mask_no_ingresante.sum()
    + mask_a_corregir.sum()
    + mask_corregir.sum()
)

if total_correcciones > 0:
    print("\n✅ Se realizaron las siguientes correcciones:")
    if mask_puntaje_vacio.sum() > 0:
        print(f"📌 {mask_puntaje_vacio.sum()} registros imputados con PUNTAJE_OBTENIDO vacio → asignado '0'")
    if (mask_nac_peru & mask_ubigeo_vacio).sum() > 0:
        print(f"📌 {(mask_nac_peru & mask_ubigeo_vacio).sum()} Peruanos sin UBIGEO → asignado '140101'")
    if mask_celular_vacio.sum() > 0:
        print(f"📌 {mask_celular_vacio.sum()} números de CELULAR generados aleatoriamente")
    if mask_no_ingresante.sum() > 0:
        print(f"📌 {mask_no_ingresante.sum()} Registros de no ingresantes con campos de ingreso corregidos")
    if mask_a_corregir.sum() > 0:
        print(f"📧 Se generaron {mask_a_corregir.sum()} correos institucionales.")
    if mask_corregir.sum() > 0:
        print(f"📧 Se generaron {mask_corregir.sum()} correos personales.")

# Aplicar las reglas fila por fila
def validar_documento_con_motivo(row):
    tipo = row["TIPO_DOCUMENTO"]
    nro = row["NRO_DOCUMENTO"]
    try:
        validador = reglas_documento_motivo.get(int(tipo), lambda x: None)
        return validador(nro)
    except:
        return " Tipo de documento inválido o error inesperado"


# Crear un dataframe solo con filas inválidas (sin modificar df original)
df_doc_obs = df.copy()
df_doc_obs["MOTIVO"] = df_doc_obs.apply(validar_documento_con_motivo, axis=1)
df_doc_obs = df_doc_obs[df_doc_obs["MOTIVO"].notnull()]

# Mostrar resumen si existen observaciones
if not df_doc_obs.empty:
    print(f"\n ❌ Se encontraron {len(df_doc_obs)} documentos inválidos:")
    display(df_doc_obs[["TIPO_DOCUMENTO", "NRO_DOCUMENTO", "MOTIVO"]])
    
for col in ["CODIGO_PROGRAMA_OPCIONES", "CODIGO_PROGRAMA_INGRESO", "CODIGO_FACULTAD_UNIDAD", "CODIGO_FACULTAD_UNIDAD_INGRESO"]:
    if col not in df.columns:
        df[col] = ""

cod_op = pd.to_numeric(df["CODIGO_PROGRAMA_OPCIONES"], errors="coerce")
df.loc[cod_op.eq(222), "CODIGO_FACULTAD_UNIDAD"] = 7

cod_in = pd.to_numeric(df["CODIGO_PROGRAMA_INGRESO"], errors="coerce")
df.loc[cod_in.eq(222), "CODIGO_FACULTAD_UNIDAD_INGRESO"] = 7


# 1. Validar que SEXO solo contenga 1 o 2

mask_sexo_invalido = ~pd.to_numeric(df['SEXO'], errors='coerce').isin([1, 2])

# Se crea el DataFrame de reporte solo si hay filas que cumplan la condición
if mask_sexo_invalido.any():
    # Se copia solo las filas inválidas para el reporte
    df_sexo_obs = df[mask_sexo_invalido].copy()
    
    # Se añade la columna con el motivo del error
    df_sexo_obs['MOTIVO'] = 'El valor es inválido.'
    
    print(f"\n❌ Se encontraron {len(df_sexo_obs)} registros con 'SEXO' inválido:")
    
    # Se muestra el reporte con las columnas más importantes
    display(df_sexo_obs[['NOMBRES', 'PRIMER_APELLIDO', 'SEXO', 'MOTIVO']])
else:
    print("✅ Campo 'SEXO' validado correctamente.")




# 3 y 4. Validar FECHA_NACIMIENTO (obligatoria, formato y mayoría de edad)
# -------------------------------------------------------------------------

def validar_fecha_nacimiento_y_edad(fecha_str):
    """
    Realiza una validación completa de la fecha de nacimiento.
    Verifica si está vacía, si tiene el formato correcto y si corresponde a un mayor de edad.
    Devuelve un mensaje de error específico o None si todo es correcto.
    """
    # Primero, se valida si el campo está vacío o es nulo.
    if pd.isna(fecha_str) or str(fecha_str).strip() == "":
        return "Fecha de nacimiento es obligatoria y está vacía."
    
    # Segundo, se intenta convertir al formato de fecha esperado.
    try:
        # Usamos pd.to_datetime que es robusto para la conversión.
        fecha_nac = pd.to_datetime(fecha_str, format='%d/%m/%Y', errors='raise')
        
        # Tercero, se calcula la edad.
        edad = (datetime.now() - fecha_nac).days / 365.25
        if edad < 18:
            return f"Postulante es menor de edad ({int(edad)} años)."
        
        # Si todas las validaciones pasan, no hay error.
        return None
        
    except ValueError:
        # Si la conversión a fecha falla, el formato es incorrecto.
        return f"El valor '{fecha_str}' tiene un formato de fecha inválido (debe ser DD/MM/AAAA)."

# Se crea el DataFrame para el reporte aplicando la función de validación.
# A diferencia del código anterior, aquí NO se usa .fillna('') porque la función ya maneja los nulos.
df_fecha_obs = df.copy()
df_fecha_obs["MOTIVO"] = df_fecha_obs["FECHA_NACIMIENTO"].apply(validar_fecha_nacimiento_y_edad)

# Se filtra para quedarse solo con las filas que tienen un motivo de error.
df_fecha_obs = df_fecha_obs[df_fecha_obs["MOTIVO"].notna()]

# Si el DataFrame de observaciones no está vacío, se muestra la alerta y el listado.
# Si está vacío, no se muestra nada.
if not df_fecha_obs.empty:
    print(f"\n❌ Se encontraron {len(df_fecha_obs)} problemas en el campo 'FECHA_NACIMIENTO':")
    display(df_fecha_obs[["NOMBRES", "PRIMER_APELLIDO", "FECHA_NACIMIENTO", "MOTIVO"]])


# 5. Informar si hay registros sin NINGÚN apellido (ni paterno ni materno)
# -------------------------------------------------------------------------
# Se convierten los NaN (nulos) a texto vacío '' ANTES de comparar.
# Esta es la forma más robusta de detectar celdas vacías.
primer_ape_vacio = df['PRIMER_APELLIDO'].fillna('').astype(str).str.strip() == ""
segundo_ape_vacio = df['SEGUNDO_APELLIDO'].fillna('').astype(str).str.strip() == ""

# Creamos la máscara para la condición:
# NO tiene PRIMER_APELLIDO Y TAMPOCO tiene SEGUNDO_APELLIDO
mask_sin_apellidos = primer_ape_vacio & segundo_ape_vacio

# Si la máscara encuentra alguna fila, muestra la alerta y el listado.
# Si no encuentra ninguna, no hace nada.
if mask_sin_apellidos.any():
    # Creamos el DataFrame para el reporte, copiando solo las filas afectadas
    df_apellidos_obs = df[mask_sin_apellidos].copy()
    
    # Añadimos el motivo de la observación para mayor claridad
    df_apellidos_obs['MOTIVO'] = 'No tiene registrado ni el primer ni el segundo apellido.'
    
    print(f"\n❌ Se encontraron {len(df_apellidos_obs)} registros sin ningún apellido. Revisar urgentemente.")
    
    # Mostramos el reporte con las columnas relevantes
    display(df_apellidos_obs[['NOMBRES', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'MOTIVO']])


# 6. Forzar FECHA_POSTULACION y FECHA_INGRESO a estar vacíos
# ----------------------------------------------------
df['FECHA_POSTULACION'] = ""
df['FECHA_INGRESO'] = ""
print("\n⚠️ Colocar la FECHA_POSTULACION y FECHA_INGRESO solo a Ingresantes de acuerdo al SIU")


#  Guardar archivo final
df.to_excel(nombre_salida, index=False)

print(f"\n✅ Archivo guardado como '{nombre_salida}'.")



✅ Se realizaron las siguientes correcciones:
📌 55 registros imputados con PUNTAJE_OBTENIDO vacio → asignado '0'
📌 263 Peruanos sin UBIGEO → asignado '140101'
📌 31 números de CELULAR generados aleatoriamente
📌 57 Registros de no ingresantes con campos de ingreso corregidos
📧 Se generaron 255 correos personales.

 ❌ Se encontraron 2 documentos inválidos:


,TIPO_DOCUMENTO,NRO_DOCUMENTO,MOTIVO
92,RUC,10456224127,Tipo de documento inválido o error inesperado
301,1,S87441827,DNI debe tener exactamente 8 dígitos



❌ Se encontraron 7 registros con 'SEXO' inválido:


,NOMBRES,PRIMER_APELLIDO,SEXO,MOTIVO
55,Juan Samuel,Villegas,No Registrado,El valor es inválido.
112,Yesenia,Ugaz,No Registrado,El valor es inválido.
221,Diego Sergio,Mendez,No Registrado,El valor es inválido.
225,Sandra Emperatriz,Perdomo,No Registrado,El valor es inválido.
254,Pedro Hernán,Mori,No Registrado,El valor es inválido.
310,Jaime Eddy,Otazú,No Registrado,El valor es inválido.
320,Edgar,Ayala,No Registrado,El valor es inválido.



❌ Se encontraron 19 problemas en el campo 'FECHA_NACIMIENTO':


,NOMBRES,PRIMER_APELLIDO,FECHA_NACIMIENTO,MOTIVO
15,Eileen,Díaz,,Fecha de nacimiento es obligatoria y está vacía.
55,Juan Samuel,Villegas,,Fecha de nacimiento es obligatoria y está vacía.
112,Yesenia,Ugaz,,Fecha de nacimiento es obligatoria y está vacía.
119,Mirla Yuriko,Nakayo,,Fecha de nacimiento es obligatoria y está vacía.
122,Ruth Melissa,Barboza,,Fecha de nacimiento es obligatoria y está vacía.
127,Quilder,Sanchez,,Fecha de nacimiento es obligatoria y está vacía.
156,ROBERTO JOSE,LEAL,,Fecha de nacimiento es obligatoria y está vacía.
165,Ruth Lizbet,García,,Fecha de nacimiento es obligatoria y está vacía.
177,Rufina Silvia,Rojas,,Fecha de nacimiento es obligatoria y está vacía.
204,Jenny Katherine,Falla,,Fecha de nacimiento es obligatoria y está vacía.



⚠️ Colocar la FECHA_POSTULACION y FECHA_INGRESO solo a Ingresantes de acuerdo al SIU

✅ Archivo guardado como '/Users/jhonvilcaranatintaya/postulantesmaestria_final.xlsx'.
